# Decision Tree Assignment Solution

## Section 1: Theoretical Questions

### Q1. What is a Decision Tree? Explain its structure (Root, Nodes, Leaves) with a real-life example.
**Answer:**
A **Decision Tree** is a non-parametric supervised machine learning algorithm used for both classification and regression tasks. It builds a flowchart-like tree structure by continually splitting a dataset into smaller, more homogeneous subsets based on feature conditions.

**Structure:**
1. **Root Node:** Represents the entire population or dataset before any splits occur. It is split into two or more child nodes using the feature that provides the highest information gain or lowest impurity reduction.
2. **Decision/Internal Nodes:** Intermediate nodes that split into further sub-nodes based on specific conditional rules.
3. **Leaf/Terminal Nodes:** End nodes that hold the final output class prediction (or continuous value) and do not split further.

**Real-Life Example:**
Deciding whether to play tennis based on weather conditions:
* **Root Node:** Check *Outlook* -> If "Overcast", **Play** (Leaf Node); if "Rainy", check *Wind*.
* **Decision Node:** Check *Wind* -> If "Strong", **Don't Play** (Leaf Node); if "Weak", **Play** (Leaf Node).

---

### Q2. Differentiate between Gini Impurity and Entropy. Which one is used by default in Scikit-learn and why?
**Answer:**
* **Gini Impurity:** Measures the probability that a randomly chosen element from the set would be incorrectly labeled if it were randomly labeled according to the distribution of labels in the subset.
  $$\text{Gini} = 1 - \sum_{i=1}^{k} p_i^2$$
* **Entropy:** Measures the degree of randomness, disorder, or uncertainty in a dataset.
  $$\text{Entropy} = -\sum_{i=1}^{k} p_i \log_2(p_i)$$

| Feature | Gini Impurity | Entropy |
| :--- | :--- | :--- |
| **Computation Speed** | Faster (avoids logarithmic calculations) | Slower (requires logarithmic calculations) |
| **Value Range (Binary)** | $0$ to $0.5$ | $0$ to $1$ |

**Default in Scikit-learn:**
`Scikit-learn` uses **Gini Impurity** (`criterion='gini'`) by default in `DecisionTreeClassifier` because it is computationally faster and yields virtually identical trees compared to Entropy in most practical applications.

---

### Q3. What is Overfitting in Decision Trees? How can we detect it using training and testing accuracy?
**Answer:**
**Overfitting** occurs when a decision tree grows too deep, capturing noise, outliers, and hyper-specific details of the training set rather than underlying general patterns.

**Detection using Accuracy:**
* **Overfitting:** Very High Training Accuracy (near 100%) paired with significantly lower Testing Accuracy.
* **Underfitting:** Low Training Accuracy and Low Testing Accuracy.
* **Good Generalization:** High Training Accuracy and High Testing Accuracy close to each other.

---

### Q4. Explain Pruning in Decision Trees. What is the difference between Pre-pruning and Post-pruning?
**Answer:**
**Pruning** is a technique used to remove non-critical branches/nodes from a decision tree to reduce model complexity and prevent overfitting.

1. **Pre-pruning (Early Stopping):** Stops tree growth *before* it perfectly fits the training set based on predefined stopping conditions (e.g., `max_depth`, `min_samples_split`, `min_samples_leaf`).
2. **Post-pruning:** Grows the decision tree to its full depth first, and then recursively prunes non-significant nodes bottom-up using techniques like Cost Complexity Pruning (`ccp_alpha`).

---

### Q5. What is Feature Importance? How can it help businesses in decision-making?
**Answer:**
**Feature Importance** measures the total reduction in criterion impurity (Gini or Entropy) brought by a feature across all splits in the tree, normalized so all feature importances sum to 1.

**Business Applications:**
1. **Targeted Campaigns:** Focus marketing efforts on key predictors (e.g., call duration, past success).
2. **Cost Savings:** Eliminate collection of irrelevant or redundant features.
3. **Strategy Optimization:** Identify key operational leverage points that drive customer conversions.


# Section 2: Practical Questions

In [ ]:
# Q6. Data Understanding
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('bank.csv')

print("Shape of dataset:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nColumn Names:\n", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

### Interpretation (Q6):
- **Shape:** 11,162 rows and 17 columns.
- **Numerical Variables:** `age`, `balance`, `day`, `duration`, `campaign`, `pdays`, `previous`.
- **Categorical Variables:** `job`, `marital`, `education`, `default`, `housing`, `loan`, `contact`, `month`, `poutcome`, `deposit` (target variable).


In [ ]:
# Q7. Data Cleaning
missing_values = df.isnull().sum()
print("Missing values per column:\n", missing_values)

duplicate_count = df.duplicated().sum()
print("\nDuplicate rows count:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates dropped.")

### Interpretation (Q7):
- **Missing Values:** 0 missing values across all columns.
- **Duplicates:** 0 duplicate rows.
- **Conclusion:** The dataset is clean and ready for preprocessing without requiring missing value imputation or deduplication.


In [ ]:
# Q8. Data Preprocessing
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=['object']).columns
df_encoded = df.copy()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("Categorical columns encoded using LabelEncoder:", list(categorical_cols))
df_encoded.head()

### Interpretation (Q8):
- **Method Used:** `LabelEncoder` was applied to convert string categorical columns into numerical values.
- **Why Encoding is Necessary:** Decision Tree algorithms in Scikit-learn require numeric inputs to compute numerical split thresholds and Gini Impurity calculations.


In [ ]:
# Q9. Feature Selection & Splitting
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['deposit'])
y = df_encoded['deposit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("X_train shape:", X_train.shape, "| X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape, "| y_test shape:", y_test.shape)

### Interpretation (Q9):
- **Train-Test Split Importance:** Splitting data into 80% training and 20% testing sets ensures the model is trained on one portion of data and evaluated independently on unseen data to accurately test generalization performance.


In [ ]:
# Q10. Model Building
from sklearn.tree import DecisionTreeClassifier

dt_model1 = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
dt_model1.fit(X_train, y_train)

print("Decision Tree Model (criterion='gini', max_depth=5) trained successfully.")

### Interpretation (Q10):
- **`max_depth` Control:** `max_depth` specifies the maximum depth (number of levels) of the decision tree. Setting `max_depth=5` serves as pre-pruning, preventing the tree from growing indefinitely and overfitting the training data.


In [ ]:
# Q11. Model Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred1 = dt_model1.predict(X_test)

acc1 = accuracy_score(y_test, y_pred1)
cm1 = confusion_matrix(y_test, y_pred1)
cr1 = classification_report(y_test, y_pred1)

print(f"Accuracy Score: {acc1:.4f}\n")
print("Confusion Matrix:\n", cm1, "\n")
print("Classification Report:\n", cr1)

### Interpretation (Q11):
- **Model Performance:** The model achieves ~80.52% testing accuracy, indicating solid overall performance for a shallow tree (`max_depth=5`).
- **Precision & Recall:** Precision for class 1 ('yes') is ~76% (76% of predicted subscribers actually subscribe), while Recall is ~87% (87% of all actual subscribers are captured).
- **Class Imbalance:** Target class distribution is balanced (~52% 'no' vs ~48% 'yes'), so there is no severe class imbalance issue.


In [ ]:
# Q12. Overfitting Check
y_train_pred1 = dt_model1.predict(X_train)

train_acc1 = accuracy_score(y_train, y_train_pred1)
test_acc1 = accuracy_score(y_test, y_pred1)

print(f"Training Accuracy: {train_acc1:.4f} ({train_acc1*100:.2f}%)")
print(f"Testing Accuracy:  {test_acc1:.4f} ({test_acc1*100:.2f}%)")

### Interpretation (Q12):
- **Status:** The training accuracy (81.50%) and testing accuracy (80.52%) are very close (gap < 1%).
- **Conclusion:** The model with `max_depth=5` is **neither overfitting nor underfitting**; it generalizes well to unseen testing data.


In [ ]:
# Q13. Pruning Experiment
dt_model2 = DecisionTreeClassifier(criterion='gini', max_depth=3, min_samples_split=20, random_state=42)
dt_model2.fit(X_train, y_train)

train_acc2 = accuracy_score(y_train, dt_model2.predict(X_train))
test_acc2 = accuracy_score(y_test, dt_model2.predict(X_test))

print(f"Model 2 Training Accuracy: {train_acc2:.4f} ({train_acc2*100:.2f}%)")
print(f"Model 2 Testing Accuracy:  {test_acc2:.4f} ({test_acc2*100:.2f}%)")
print(f"\nModel 1 Testing Accuracy: {test_acc1:.4f} ({test_acc1*100:.2f}%)")

### Interpretation (Q13):
- **Comparison:** Model 1 (`max_depth=5`) achieved **80.52%** test accuracy, whereas Model 2 (`max_depth=3, min_samples_split=20`) achieved **76.53%** test accuracy.
- **Verdict:** **Model 1 is better.** Restricting tree depth to 3 causes slight underfitting because it oversimplifies decision boundaries.


In [ ]:
# Q14. Feature Importance
import matplotlib.pyplot as plt

importances = dt_model1.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

print("Top 5 Important Features:")
print(feat_imp.head(5))

# Plot top 5 feature importances
plt.figure(figsize=(8, 5))
feat_imp.head(5).plot(kind='barh', color='teal')
plt.title('Top 5 Feature Importances (Model 1)')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Interpretation (Q14):
- **Top Influential Feature:** `duration` (call duration) is by far the most influential feature (~59.5% importance), followed by `contact`, `pdays`, `housing`, and `month`.


In [ ]:
# Q15. Business Insights
print("Business Insights & Strategic Recommendations:")

### Business Insights (Q15):

1. **Which type of customers are more likely to say "yes"?**
   - **High Engagement:** Customers who stay on outreach calls longer (`duration`).
   - **Past Success:** Customers who responded positively in prior campaigns (`poutcome` / `pdays`).
   - **No Housing Debt:** Customers without active housing loans (`housing=no`) demonstrate higher conversion rates.

2. **What strategy should a bank use?**
   - **Improve Pitch Quality:** Train representatives to engage clients longer during sales calls, as call duration is the single strongest predictor of deposit conversion.
   - **Prioritize Warm Leads:** Focus sales drives on existing customers with positive previous campaign records (`poutcome`).
   - **Timing & Filtering:** Align marketing outreach with high-performing campaign months and reduce cold-calling customers with high debt burdens.
